# LeetCode 1342 — Number of Steps to Reduce a Number to Zero bằng Python

**Link đề:** https://leetcode.com/problems/number-of-steps-to-reduce-a-number-to-zero/
**Độ khó:** Easy · **Chủ đề:** Math, Bit Manipulation (mô phỏng → công thức nhị phân)

```python
class Solution:
    def numberOfSteps(self, num: int) -> int:
        ...
```

## Mục tiêu học
1. Viết **mô phỏng** (simulation) đúng theo đề với `while`, `%`, `//`.
2. Hiểu số chẵn/lẻ dưới góc nhìn **nhị phân**: chia 2 = dịch phải `>> 1`, trừ 1 khi lẻ = xoá bit cuối.
3. Rút ra **công thức** `bit_length + số bit 1 − 1` và chứng minh vì sao đúng.
4. So sánh 4 lời giải: mô phỏng, toán tử bit, công thức `bin()`, đệ quy.
5. Trace, Time/Space, lỗi hay gặp (`num = 0`, dùng `/`), bài tập biến thể.

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

Cho số nguyên `num`, trả về **số bước** để biến nó thành `0`.

Mỗi bước: nếu số hiện tại **chẵn** thì chia cho 2, nếu **lẻ** thì trừ đi 1.

| `num` | Output | Các bước |
|---:|---:|---|
| `14` | `6` | 14 → 7 → 6 → 3 → 2 → 1 → 0 |
| `8` | `4` | 8 → 4 → 2 → 1 → 0 |
| `123` | `12` | (sẽ trace ở mục 6) |

**Constraints:** `0 <= num <= 10^6`.

Câu hỏi trước khi code:
- `num = 0` cần bao nhiêu bước?
- Có thể tính số bước **không cần mô phỏng** không?

## 2. Nền tảng Python

### 2.1. Chẵn/lẻ với `%` và phép chia nguyên `//`

- `x % 2 == 0` ⇔ `x` chẵn.
- `x // 2` chia lấy phần nguyên (kết quả `int`). Khác `x / 2` cho `float` (`7 / 2 = 3.5`, `8 / 2 = 4.0`).

In [1]:
for x in [14, 7, 0, 1]:
    print(f"{x}: chẵn? {x % 2 == 0} | x // 2 = {x // 2} | x / 2 = {x / 2}")

14: chẵn? True | x // 2 = 7 | x / 2 = 7.0
7: chẵn? False | x // 2 = 3 | x / 2 = 3.5
0: chẵn? True | x // 2 = 0 | x / 2 = 0.0
1: chẵn? False | x // 2 = 0 | x / 2 = 0.5


### 2.2. Biểu diễn nhị phân: `bin()`, `bit_length()`, `count('1')`

- `bin(14)` → `'0b1110'`.
- `(14).bit_length()` → `4`: số bit cần để viết `14` (không tính tiền tố, không có số 0 đứng đầu). `(0).bit_length()` → `0`.
- `bin(14).count('1')` → `3`: số bit 1. (Python 3.10+ còn có `(14).bit_count()`.)

In [2]:
for x in [14, 8, 123, 1, 0]:
    print(f"{x:>4}: bin={bin(x):>10}  bit_length={x.bit_length()}  số bit 1={bin(x).count('1')}")

  14: bin=    0b1110  bit_length=4  số bit 1=3
   8: bin=    0b1000  bit_length=4  số bit 1=1
 123: bin= 0b1111011  bit_length=7  số bit 1=6
   1: bin=       0b1  bit_length=1  số bit 1=1
   0: bin=       0b0  bit_length=0  số bit 1=0


### 2.3. Toán tử bit: `& 1` và `>> 1`

- `x & 1` lấy **bit cuối**: `1` nếu lẻ, `0` nếu chẵn (nhanh hơn `% 2` một chút, ý nghĩa giống hệt với số không âm).
- `x >> 1` **dịch phải 1 bit** = bỏ bit cuối = `x // 2` (với số không âm).
- `x - 1` khi `x` lẻ: bit cuối `1` thành `0` (các bit khác giữ nguyên). Tương đương `x ^ 1` (XOR với 1) khi `x` lẻ.

In [3]:
x = 14
print(f"{x} = {x:b}, x & 1 = {x & 1}, x >> 1 = {x >> 1} = {x >> 1:b}")
x = 7
print(f"{x} = {x:b}, x & 1 = {x & 1}, x - 1 = {x - 1} = {x - 1:b}, x ^ 1 = {x ^ 1}")

14 = 1110, x & 1 = 0, x >> 1 = 7 = 111
7 = 111, x & 1 = 1, x - 1 = 6 = 110, x ^ 1 = 6


### 2.4. Tự đếm bit 1 không dùng chuỗi: mẹo `x & (x - 1)`

`x - 1` lật bit 1 **thấp nhất** thành 0 và mọi bit 0 phía sau nó thành 1. Vì vậy `x & (x - 1)` **xoá đúng bit 1 thấp nhất** của `x`. Lặp cho tới khi `x == 0`, số lần lặp = số bit 1 (thuật toán Brian Kernighan).

Ví dụ `12 = 1100₂`: `12 & 11 = 1100 & 1011 = 1000 = 8`; `8 & 7 = 0`. Hai lần → 2 bit 1.

In [4]:
def popcount(x: int) -> int:
    count = 0
    while x:
        print(f"    x = {x:>8b} -> x & (x-1) = {x & (x - 1):>8b}")
        x &= x - 1
        count += 1
    return count

for x in [12, 123]:
    print(f"x = {x}:")
    print("  số bit 1 =", popcount(x), "| bin().count('1') =", bin(x).count('1'))

x = 12:
    x =     1100 -> x & (x-1) =     1000
    x =     1000 -> x & (x-1) =        0
  số bit 1 = 2 | bin().count('1') = 2
x = 123:
    x =  1111011 -> x & (x-1) =  1111010
    x =  1111010 -> x & (x-1) =  1111000
    x =  1111000 -> x & (x-1) =  1110000
    x =  1110000 -> x & (x-1) =  1100000
    x =  1100000 -> x & (x-1) =  1000000
    x =  1000000 -> x & (x-1) =        0
  số bit 1 = 6 | bin().count('1') = 6


Trong f-string, `{x:b}` in `x` dạng nhị phân không có tiền tố `0b`.

## 3. Cách trực giác / baseline — mô phỏng đúng như đề

Lặp trong khi `num > 0`: chẵn thì `num //= 2`, lẻ thì `num -= 1`, mỗi lần tăng `steps`.

In [5]:
def steps_simulate(num: int) -> int:
    steps = 0
    while num > 0:
        if num % 2 == 0:
            num //= 2
        else:
            num -= 1
        steps += 1
    return steps

for num in [14, 8, 123, 0, 1]:
    print(num, "->", steps_simulate(num))

14 -> 6
8 -> 4
123 -> 12
0 -> 0
1 -> 1


### 3.1. Mô phỏng có nhanh không?

Mỗi **hai** bước liên tiếp, số ít nhất giảm một nửa (lẻ → trừ 1 thành chẵn → chia 2). Vậy số bước ≤ `2 · log2(num)` ≈ `2 × 20 = 40` với `num <= 10^6`. Rất nhanh: `O(log num)`.

Vậy "nút thắt" ở đây không phải tốc độ, mà là câu hỏi hay hơn: **số bước có công thức đóng không?** Hãy nhìn các bước dưới dạng nhị phân.

In [6]:
num = 14
while num > 0:
    action = "chẵn -> chia 2 (bỏ bit cuối 0)" if num % 2 == 0 else "lẻ  -> trừ 1 (bit cuối 1 thành 0)"
    print(f"{num:>3} = {num:>5b}  {action}")
    num = num // 2 if num % 2 == 0 else num - 1
print(f"{num:>3} = {num:>5b}")

 14 =  1110  chẵn -> chia 2 (bỏ bit cuối 0)
  7 =   111  lẻ  -> trừ 1 (bit cuối 1 thành 0)
  6 =   110  chẵn -> chia 2 (bỏ bit cuối 0)
  3 =    11  lẻ  -> trừ 1 (bit cuối 1 thành 0)
  2 =    10  chẵn -> chia 2 (bỏ bit cuối 0)
  1 =     1  lẻ  -> trừ 1 (bit cuối 1 thành 0)
  0 =     0


In [7]:
# Bảng nhỏ: số bước so với số bit -- quan sát số bước nằm giữa L và 2L - 1
print(" num | nhị phân | L | bước | 2L-1")
for num in [1, 2, 3, 4, 7, 8, 15, 16, 31, 100]:
    L = num.bit_length()
    print(f"{num:>4} | {num:>8b} | {L} | {steps_simulate(num):>4} | {2 * L - 1:>4}")

 num | nhị phân | L | bước | 2L-1
   1 |        1 | 1 |    1 |    1
   2 |       10 | 2 |    2 |    3
   3 |       11 | 2 |    3 |    3
   4 |      100 | 3 |    3 |    5
   7 |      111 | 3 |    5 |    5
   8 |     1000 | 4 |    4 |    7
  15 |     1111 | 4 |    7 |    7
  16 |    10000 | 5 |    5 |    9
  31 |    11111 | 5 |    9 |    9
 100 |  1100100 | 7 |    9 |   13


Quan sát: lũy thừa của 2 (`1000...0`) cho số bước **ít nhất** = `L`; số toàn bit 1 (`111...1`) cho **nhiều nhất** = `2L − 1`. Gợi ý mạnh: số bước phụ thuộc **số bit** và **số bit 1**.

## 4. Insight — đếm theo bit

Nhìn bảng trên với `14 = 1110₂`:
- Bit cuối là **0** → chia 2 = **xoá bit cuối** (1 bước).
- Bit cuối là **1** → trừ 1 biến nó thành 0 (1 bước), rồi chia 2 xoá nó (thêm 1 bước) — trừ khi đó là **bit cuối cùng còn lại** (số = 1 → trừ 1 thành 0, xong).

Vậy:
- Mỗi bit **0** tốn 1 bước.
- Mỗi bit **1** tốn 2 bước, **riêng bit 1 cao nhất** chỉ tốn 1 bước.

Tổng = `(số bit 0) + 2·(số bit 1) − 1` = `bit_length + số bit 1 − 1` (với `num > 0`). Với `num = 0`: 0 bước.

Kiểm tra `14 = 1110₂`: `bit_length = 4`, số bit 1 = 3 ⇒ `4 + 3 − 1 = 6` ✔.

In [8]:
for num in [14, 8, 123]:
    L, ones = num.bit_length(), bin(num).count('1')
    print(f"{num} = {num:b}: bit_length={L}, bit 1={ones} -> {L} + {ones} - 1 = {L + ones - 1} | mô phỏng: {steps_simulate(num)}")

14 = 1110: bit_length=4, bit 1=3 -> 4 + 3 - 1 = 6 | mô phỏng: 6
8 = 1000: bit_length=4, bit 1=1 -> 4 + 1 - 1 = 4 | mô phỏng: 4
123 = 1111011: bit_length=7, bit 1=6 -> 7 + 6 - 1 = 12 | mô phỏng: 12


## 5. Lời giải chính — công thức bit, giảng từng đoạn

**Đoạn 1:** nếu `num == 0` → trả `0` (công thức sẽ cho `0 + 0 − 1 = −1`, sai!).

**Đoạn 2:** `L = num.bit_length()` — số bit.

**Đoạn 3:** `ones = bin(num).count('1')` — số bit 1.

**Đoạn 4:** trả `L + ones − 1`.

In [9]:
def steps_formula(num: int) -> int:
    if num == 0:
        return 0
    return num.bit_length() + bin(num).count('1') - 1

for num in [14, 8, 123, 0, 1, 2, 3]:
    print(num, "->", steps_formula(num))

14 -> 6
8 -> 4
123 -> 12
0 -> 0
1 -> 1
2 -> 2
3 -> 3


### 5.1. Phiên bản mô phỏng bằng toán tử bit (rất hay dùng trong phỏng vấn)

In [10]:
def steps_bitwise(num: int) -> int:
    steps = 0
    while num:
        if num & 1:
            num -= 1        # hoặc num ^= 1
        else:
            num >>= 1
        steps += 1
    return steps

print([steps_bitwise(x) for x in [14, 8, 123, 0]])

[6, 4, 12, 0]


## 6. TRACE từng bước

Trace mô phỏng `num = 123 = 1111011₂`:

| bước | num | nhị phân | chẵn/lẻ | hành động | num mới |
|---|---|---|---|---|---|
| 1 | 123 | 1111011 | lẻ | −1 | 122 |
| 2 | 122 | 1111010 | chẵn | /2 | 61 |
| 3 | 61 | 111101 | lẻ | −1 | 60 |
| 4 | 60 | 111100 | chẵn | /2 | 30 |
| 5 | 30 | 11110 | chẵn | /2 | 15 |
| 6 | 15 | 1111 | lẻ | −1 | 14 |
| 7 | 14 | 1110 | chẵn | /2 | 7 |
| 8 | 7 | 111 | lẻ | −1 | 6 |
| 9 | 6 | 110 | chẵn | /2 | 3 |
| 10 | 3 | 11 | lẻ | −1 | 2 |
| 11 | 2 | 10 | chẵn | /2 | 1 |
| 12 | 1 | 1 | lẻ | −1 | 0 |

Công thức: `bit_length = 7`, bit 1 = 6 ⇒ `7 + 6 − 1 = 12` ✔.

In [11]:
def trace_steps(num: int) -> int:
    steps = 0
    print(f"num = {num} = {num:b}")
    while num > 0:
        if num % 2 == 0:
            new, act = num // 2, "chẵn -> // 2"
        else:
            new, act = num - 1, "lẻ   -> - 1 "
        steps += 1
        print(f"  bước {steps:>2}: {num:>4} ({num:>8b}) {act} -> {new}")
        num = new
    print("  tổng số bước:", steps)
    return steps

trace_steps(123)
print()
trace_steps(8)

num = 123 = 1111011
  bước  1:  123 ( 1111011) lẻ   -> - 1  -> 122
  bước  2:  122 ( 1111010) chẵn -> // 2 -> 61
  bước  3:   61 (  111101) lẻ   -> - 1  -> 60
  bước  4:   60 (  111100) chẵn -> // 2 -> 30
  bước  5:   30 (   11110) chẵn -> // 2 -> 15
  bước  6:   15 (    1111) lẻ   -> - 1  -> 14
  bước  7:   14 (    1110) chẵn -> // 2 -> 7
  bước  8:    7 (     111) lẻ   -> - 1  -> 6
  bước  9:    6 (     110) chẵn -> // 2 -> 3
  bước 10:    3 (      11) lẻ   -> - 1  -> 2
  bước 11:    2 (      10) chẵn -> // 2 -> 1
  bước 12:    1 (       1) lẻ   -> - 1  -> 0
  tổng số bước: 12

num = 8 = 1000
  bước  1:    8 (    1000) chẵn -> // 2 -> 4
  bước  2:    4 (     100) chẵn -> // 2 -> 2
  bước  3:    2 (      10) chẵn -> // 2 -> 1
  bước  4:    1 (       1) lẻ   -> - 1  -> 0
  tổng số bước: 4


4

## 7. Code nộp LeetCode

### Bản mô phỏng (dễ hiểu nhất)

In [12]:
class Solution:
    def numberOfSteps(self, num: int) -> int:
        steps = 0
        while num > 0:
            if num % 2 == 0:
                num //= 2
            else:
                num -= 1
            steps += 1
        return steps

print([Solution().numberOfSteps(x) for x in [14, 8, 123]])

[6, 4, 12]


### Bản công thức `O(1)` thao tác (gọi hàm có sẵn)

In [13]:
class SolutionFormula:
    def numberOfSteps(self, num: int) -> int:
        return num.bit_length() + bin(num).count('1') - 1 if num else 0

print([SolutionFormula().numberOfSteps(x) for x in [14, 8, 123, 0]])

[6, 4, 12, 0]


Ở đây là **biểu thức điều kiện** `A if num else 0`: `num` khác 0 được coi là đúng (truthy).

### Cách 4 — đệ quy (recursion)

Định nghĩa trực tiếp theo đề: `f(0) = 0`; `f(n) = 1 + f(n // 2)` nếu chẵn; `f(n) = 1 + f(n − 1)` nếu lẻ. Hàm **gọi lại chính nó** với input nhỏ hơn, dừng ở **trường hợp cơ sở** `n == 0`.

In [14]:
def steps_recursive(num: int) -> int:
    if num == 0:                                 # trường hợp cơ sở
        return 0
    if num % 2 == 0:
        return 1 + steps_recursive(num // 2)
    return 1 + steps_recursive(num - 1)

class SolutionRecursive:
    def numberOfSteps(self, num: int) -> int:
        if num == 0:
            return 0
        return 1 + self.numberOfSteps(num // 2 if num % 2 == 0 else num - 1)

print([steps_recursive(x) for x in [14, 8, 123, 0]])
print([SolutionRecursive().numberOfSteps(x) for x in [14, 8, 123, 0]])

[6, 4, 12, 0]
[6, 4, 12, 0]


Độ sâu đệ quy ≤ ~40 với `num <= 10^6`, an toàn (giới hạn mặc định của Python khoảng 1000).

### Bảng so sánh

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Mô phỏng `%`, `//` | làm đúng như đề | `O(log num)` | `O(1)` | dễ nhất |
| 2. Mô phỏng bit `&`, `>>` | như trên, toán tử bit | `O(log num)` | `O(1)` | hay hỏi phỏng vấn |
| 3. Công thức | `bit_length + popcount − 1` | `O(log num)`* | `O(log num)` (chuỗi `bin`) | ngắn, cần chứng minh |
| 4. Đệ quy | định nghĩa đệ quy | `O(log num)` | `O(log num)` stack | luyện tư duy đệ quy |

*`bit_length` là `O(1)` với số nhỏ; `bin(...).count` duyệt `O(log num)` ký tự. Với `int.bit_count()` (3.10+) cũng rất nhanh.

## 8. Vì sao công thức đúng?

Cho `num > 0` có biểu diễn nhị phân `b_{L-1} ... b_1 b_0` với `b_{L-1} = 1` (bit cao nhất), `L = bit_length`.

Xét quá trình: tại mỗi thời điểm, bit cuối quyết định hành động.
- Bit cuối `0` (và số > 0): chia 2 = xoá bit đó. **1 bước** cho bit 0 này.
- Bit cuối `1`, và còn bit phía trước: trừ 1 (bit thành 0), rồi chia 2 (xoá). **2 bước** cho bit 1 này.
- Bit cuối `1` và là bit duy nhất còn lại (số = 1): trừ 1 → 0. **1 bước**.

Mỗi bit bị xoá đúng một lần, theo thứ tự từ phải sang trái; các bit khác không đổi. Vì vậy:

`steps = (#bit 0) · 1 + (#bit 1 − 1) · 2 + 1 = (L − ones) + 2·ones − 2 + 1 = L + ones − 1`. ∎

### 8.1. Độ phức tạp

Gọi `L = ⌊log2 num⌋ + 1` (số bit, ≤ 20 với `num <= 10^6`).
- Mô phỏng: `O(L)` thời gian (≤ `2L − 1` bước), `O(1)` bộ nhớ.
- Công thức: `O(L)` để tạo chuỗi `bin` và đếm; `O(L)` bộ nhớ tạm.

In [15]:
# Kiểm chứng phân tích "chi phí theo từng bit" cho một số cụ thể
def cost_per_bit(num: int) -> list[tuple[int, int, int]]:
    bits = format(num, 'b')[::-1]           # đảo để duyệt từ bit thấp (vị trí 0)
    L = len(bits)
    costs = []
    for pos, ch in enumerate(bits):
        if ch == '0':
            c = 1
        elif pos == L - 1:
            c = 1                           # bit 1 cao nhất: chỉ cần trừ 1
        else:
            c = 2                           # bit 1 khác: trừ 1 rồi chia 2
        costs.append((pos, int(ch), c))
    return costs

num = 123
costs = cost_per_bit(num)
for pos, bit, c in costs:
    print(f"  bit vị trí {pos}: {bit} -> {c} bước")
print("tổng chi phí:", sum(c for _, _, c in costs), "| mô phỏng:", steps_simulate(num))
for x in range(1, 5000):
    assert sum(c for _, _, c in cost_per_bit(x)) == steps_simulate(x)
print("phân tích theo bit đúng cho mọi x = 1..4999")

  bit vị trí 0: 1 -> 2 bước
  bit vị trí 1: 1 -> 2 bước
  bit vị trí 2: 0 -> 1 bước
  bit vị trí 3: 1 -> 2 bước
  bit vị trí 4: 1 -> 2 bước
  bit vị trí 5: 1 -> 2 bước
  bit vị trí 6: 1 -> 1 bước
tổng chi phí: 12 | mô phỏng: 12
phân tích theo bit đúng cho mọi x = 1..4999


## 9. Bộ test

Mẫu đề, biên (`0`, `1`, `10^6`), lũy thừa của 2, số dạng `2^k − 1` (toàn bit 1), và **đối chiếu** mọi cách trên toàn bộ `0..10^5` (và vài số lớn).

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<14} PASS -> {actual}")

sol = Solution()
check("ví dụ 1", sol.numberOfSteps(14), 6)
check("ví dụ 2", sol.numberOfSteps(8), 4)
check("ví dụ 3", sol.numberOfSteps(123), 12)
check("num = 0", sol.numberOfSteps(0), 0)
check("num = 1", sol.numberOfSteps(1), 1)
check("2^10", sol.numberOfSteps(1024), 11)        # 10 lần chia + 1 lần trừ
check("2^10 - 1", sol.numberOfSteps(1023), 19)    # 10 bit 1: 10 + 10 - 1
check("10^6", sol.numberOfSteps(10**6), steps_simulate(10**6))

ví dụ 1        PASS -> 6
ví dụ 2        PASS -> 4
ví dụ 3        PASS -> 12
num = 0        PASS -> 0
num = 1        PASS -> 1
2^10           PASS -> 11
2^10 - 1       PASS -> 19
10^6           PASS -> 26


In [17]:
methods = [steps_formula, steps_bitwise, steps_recursive,
           sol.numberOfSteps, SolutionFormula().numberOfSteps, SolutionRecursive().numberOfSteps]
for num in list(range(0, 100001)) + [999999, 1000000, 524287, 524288]:
    expected = steps_simulate(num)
    for f in methods:
        assert f(num) == expected, (f, num)
print("Mọi cách khớp mô phỏng trên 0..100000 và các số biên lớn")
print("số bước lớn nhất với num <= 10^6:", max(steps_formula(x) for x in range(10**6 + 1)))

Mọi cách khớp mô phỏng trên 0..100000 và các số biên lớn
số bước lớn nhất với num <= 10^6: 38


## 10. Lỗi người mới hay gặp

### Lỗi 1: Dùng `/` thay vì `//`

`num = num / 2` biến `num` thành `float`. Với số nhỏ thì vẫn ra đúng số bước, nhưng kiểu dữ liệu sai và với số rất lớn sẽ mất chính xác. Luôn dùng `//` cho số nguyên.

In [18]:
num = 14
num = num / 2
print(num, type(num))
big = 2**60 + 1
print("float mất chính xác:", int(big / 1) == big, "| // vẫn đúng:", big // 1 == big)

7.0 <class 'float'>
float mất chính xác: False | // vẫn đúng: True


### Lỗi 2: Công thức quên trường hợp `num = 0`

In [19]:
def wrong_formula(num):
    return num.bit_length() + bin(num).count('1') - 1

print("num=0 -> sai:", wrong_formula(0), "| đúng:", steps_formula(0))

num=0 -> sai: -1 | đúng: 0


### Lỗi 3: Đếm bit 0 / bit 1 trên chuỗi `bin()` mà quên tiền tố `'0b'`

`bin(14) = '0b1110'`: đếm `'0'` sẽ ra `2` (tính cả số 0 của `'0b'`), không phải `1`. Dùng `bin(x)[2:]` hoặc `format(x, 'b')`.

### Lỗi 4: Điều kiện vòng lặp `while num >= 0` → vòng lặp vô hạn

Khi `num = 0`: chẵn → `0 // 2 = 0` → mãi mãi. Phải là `while num > 0`.

In [20]:
print("đếm '0' trong bin(14):", bin(14).count('0'), "| đúng:", bin(14)[2:].count('0'))

def wrong_loop_guarded(num, limit=10):
    steps = 0
    while num >= 0 and steps < limit:          # thêm limit để demo không treo máy
        num = num // 2 if num % 2 == 0 else num - 1
        steps += 1
    return steps

print("while num >= 0 với num=2 chạy tới giới hạn:", wrong_loop_guarded(2), "bước (đáng lẽ 2)")

đếm '0' trong bin(14): 2 | đúng: 1
while num >= 0 với num=2 chạy tới giới hạn: 10 bước (đáng lẽ 2)


## 11. Mẹo nhận dạng pattern

- "Chẵn thì chia 2, lẻ thì trừ 1/cộng 1" ⇒ nghĩ ngay **nhị phân**: chia 2 = `>> 1`, kiểm lẻ = `& 1`.
- Quá trình mỗi bước giảm một nửa ⇒ số bước `O(log n)`, mô phỏng là đủ.
- Muốn công thức đóng ⇒ đếm "chi phí" theo **từng bit** (bit 0 tốn bao nhiêu, bit 1 tốn bao nhiêu).
- `bit_length()`, `bin(x).count('1')`, `bit_count()` là bộ công cụ chuẩn.

## 12. Biến thể

1. **Chỉ được chia 2 khi chẵn, và trừ 1 hoặc cộng 1 khi lẻ** — LeetCode 397 Integer Replacement (tối thiểu bước tới 1): cần tham lam theo 2 bit cuối.
2. **LeetCode 1404 — Number of Steps to Reduce a Number in Binary Representation to One**: input là chuỗi nhị phân, lẻ thì **cộng** 1.
3. **Chia 3 nếu chia hết, ngược lại trừ 1**: tổng quát hoá sang cơ số 3.

In [21]:
def steps_base3(num: int) -> int:
    steps = 0
    while num > 0:
        num = num // 3 if num % 3 == 0 else num - 1
        steps += 1
    return steps

def steps_binary_string_to_one(s: str) -> int:
    # LeetCode 1404: chẵn chia 2, lẻ cộng 1, tới khi bằng 1
    num, steps = int(s, 2), 0                  # int(s, 2): đọc chuỗi nhị phân
    while num != 1:
        num = num // 2 if num % 2 == 0 else num + 1
        steps += 1
    return steps

print("cơ số 3, 10:", steps_base3(10))
print("1404, '1101':", steps_binary_string_to_one("1101"))

cơ số 3, 10: 4
1404, '1101': 6


## 13. Bài tập tự luyện (TODO — tăng dần)

**Bài 1 (tính tay):** Số bước cho `num = 37`? Làm bằng cả mô phỏng và công thức.

**Bài 2 (đoán output):** `bin(37)`, `(37).bit_length()`, `37 >> 1`, `37 & 1`.

**Bài 3 (tự viết):** `exercise_steps(num)` bằng toán tử bit `&` và `>>` (không dùng `%`, `//`).

**Bài 4 (tự viết):** `exercise_steps_formula(num)` — công thức, nhớ `num = 0`.

**Bài 5 (biến thể):** `exercise_steps_base3(num)` — chia 3 nếu chia hết, ngược lại trừ 1. Đếm số bước.

**Bài 6 (mini challenge):** `exercise_max_steps(limit)` — trả về `(số, số_bước)` có số bước **lớn nhất** với `0 <= số <= limit` (nếu hoà, lấy số nhỏ nhất). Gợi ý: số nào có nhiều bit 1 và dài nhất?

In [22]:
def exercise_steps(num: int) -> int:
    # TODO: dùng num & 1 và num >> 1
    pass

def exercise_steps_formula(num: int) -> int:
    # TODO
    pass

# assert exercise_steps(14) == 6 and exercise_steps(0) == 0
# assert exercise_steps_formula(123) == 12 and exercise_steps_formula(0) == 0

In [23]:
def exercise_steps_base3(num: int) -> int:
    # TODO
    pass

def exercise_max_steps(limit: int) -> tuple[int, int]:
    # TODO
    pass

# assert exercise_steps_base3(10) == 4      # 10 -> 9 -> 3 -> 1 -> 0
# assert exercise_max_steps(10) == (7, 5)

## 14. Gợi ý 3 mức

**Bài 1** — (1) `37 = 100101₂`. (2) `bit_length = 6`, số bit 1 = 3. (3) Công thức `6 + 3 − 1`.

**Bài 3** — (1) `while num:`. (2) `if num & 1: num -= 1`. (3) `else: num >>= 1`; luôn `steps += 1`.

**Bài 4** — (1) `if num == 0: return 0`. (2) `num.bit_length()`. (3) `bin(num).count('1')`.

**Bài 5** — (1) Copy cấu trúc mô phỏng. (2) `% 3 == 0` thay cho chẵn. (3) `// 3`.

**Bài 6** — (1) Duyệt mọi số tới `limit`, tính số bước bằng công thức. (2) Giữ `best` theo `(số_bước, -số)` lớn nhất. (3) Hoặc nghĩ: số dạng `2^k − 1` lớn nhất `<= limit` thường thắng.

## 15. Đáp án tham khảo

**Bài 1:** `37 → 36 → 18 → 9 → 8 → 4 → 2 → 1 → 0`: **8** bước; công thức `6 + 3 − 1 = 8` ✔.

**Bài 2:** `'0b100101'`, `6`, `18`, `1`.

In [24]:
def answer_steps(num):
    steps = 0
    while num:
        if num & 1:
            num -= 1
        else:
            num >>= 1
        steps += 1
    return steps

def answer_steps_formula(num):
    return 0 if num == 0 else num.bit_length() + bin(num).count('1') - 1

def answer_steps_base3(num):
    steps = 0
    while num > 0:
        num = num // 3 if num % 3 == 0 else num - 1
        steps += 1
    return steps

def answer_max_steps(limit):
    best_num, best_steps = 0, 0
    for x in range(limit + 1):
        s = answer_steps_formula(x)
        if s > best_steps:                       # chỉ cập nhật khi lớn hơn hẳn -> giữ số nhỏ nhất khi hoà
            best_num, best_steps = x, s
    return best_num, best_steps

assert steps_simulate(37) == 8 == answer_steps_formula(37)
assert bin(37) == '0b100101' and (37).bit_length() == 6 and 37 >> 1 == 18 and 37 & 1 == 1
assert answer_steps(14) == 6 and answer_steps(0) == 0
assert answer_steps_formula(123) == 12 and answer_steps_formula(0) == 0
assert answer_steps_base3(10) == 4
assert answer_max_steps(10) == (7, 5)
for x in range(3000):
    assert answer_steps(x) == answer_steps_formula(x) == steps_simulate(x)
print("max steps tới 10^6:", answer_max_steps(10**6))
print("Tất cả đáp án PASS!")

max steps tới 10^6: (786431, 38)
Tất cả đáp án PASS!


## 16. Template nhớ nhanh + checklist submit

```python
class Solution:
    def numberOfSteps(self, num: int) -> int:
        steps = 0
        while num > 0:                 # > 0, không phải >= 0
            if num % 2 == 0:
                num //= 2              # //, không phải /
            else:
                num -= 1
            steps += 1
        return steps
```

Công thức: `0 if num == 0 else num.bit_length() + bin(num).count('1') - 1`.

**Checklist:**
- [ ] `num = 0` trả `0`.
- [ ] Dùng `//` (hoặc `>>`) để giữ kiểu `int`.
- [ ] Điều kiện vòng lặp `num > 0`.
- [ ] Nếu dùng công thức: đếm bit trên `bin(x)[2:]` khi đếm số `0`.